# 00. Khởi tạo môi trường Google Colab & Kiểm tra cấu hình (P0)

**Đề tài:** Gắn nhãn các đoạn tweet có nội dung phân biệt chủng tộc, phân biệt giới tính (*Twitter Sentiment / Hate Speech Binary Classification*)  
**Mục tiêu notebook:**
1. Thiết lập cell `bootstrap` chuẩn giúp notebook chạy tự động trên cả **Google Colab** (Mount Google Drive, cài đặt thư viện, tải tài nguyên NLTK) và **máy cục bộ (Local)**.
2. Cố định `seed = 42` trên toàn bộ thư viện (`random`, `numpy`, `sklearn`, `torch`) để đảm bảo tính tái lập.
3. Kiểm tra đường dẫn dự án `PROJECT_DIR`, cấu hình `config.yaml`, phiên bản `scikit-learn`, `torch` và trạng thái GPU.

In [1]:
# === CELL BOOTSTRAP CHUẨN CHO MỌI NOTEBOOK TRONG DỰ ÁN ===
import os, sys, subprocess
from pathlib import Path

os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("OMP_NUM_THREADS", "1")
REPO_URL = "https://github.com/devtitcun888/duan-tweet-hate.git"

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    drive_mounted = False
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        drive_mounted = True
    except Exception as e:
        print(f"[BOOTSTRAP] Không mount Google Drive ({e}), dùng bộ nhớ phiên Colab /content.")
    for candidate in [
        Path("/content/drive/MyDrive/duan-tweet-hate"),
        Path("/content/drive/MyDrive/tweet-hate"),
        Path("/content/drive/MyDrive/DOAN_COLAB"),
        Path("/content/duan-tweet-hate"),
        Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd(),
    ]:
        if (candidate / "config.yaml").exists():
            PROJECT_DIR = candidate
            break
    else:
        PROJECT_DIR = Path("/content/drive/MyDrive/duan-tweet-hate") if drive_mounted else Path("/content/duan-tweet-hate")
        if not (PROJECT_DIR / "config.yaml").exists():
            print(f"[BOOTSTRAP] Đang tải dự án từ {REPO_URL} vào {PROJECT_DIR}...")
            subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
    req_file = PROJECT_DIR / "requirements.txt"
    if req_file.exists():
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(req_file)], check=False)
    import nltk
    for pkg in ["stopwords", "wordnet", "omw-1.4"]:
        nltk.download(pkg, quiet=True)
else:
    PROJECT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

os.chdir(PROJECT_DIR)
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from src.utils import set_seed, load_config, mask_profanity
set_seed(42)
CFG = load_config()
print(f"[BOOTSTRAP] PROJECT_DIR = {PROJECT_DIR}")


[BOOTSTRAP] IN_COLAB = False
[BOOTSTRAP] PROJECT_DIR = D:\TITKUL\DOAN_COLAB


In [2]:
import sklearn
import pandas as pd
import numpy as np
import torch

print("=== THÔNG TIN MÔI TRƯỜNG & PHẦN CỨNG ===")
print(f"Python version      : {sys.version.split()[0]}")
print(f"scikit-learn version: {sklearn.__version__}")
print(f"pandas version      : {pd.__version__}")
print(f"numpy version       : {np.__version__}")
print(f"PyTorch version     : {torch.__version__}")
print(f"CUDA GPU available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device          : {torch.cuda.get_device_name(0)}")
print("\n=== CẤU HÌNH DỰ ÁN (config.yaml) ===")
print(CFG)
print("\n=== KIỂM TRA HÀM CHE TỪ TỤC (mask_profanity) ===")
sample_text = "This racist bigot called her a bitch and used the word fuck."
print("Trước khi che:", mask_profanity(sample_text))

=== THÔNG TIN MÔI TRƯỜNG & PHẦN CỨNG ===
Python version      : 3.14.6
scikit-learn version: 1.9.1
pandas version      : 3.1.1
numpy version       : 2.5.5
PyTorch version     : 2.15.0+cpu
CUDA GPU available  : False

=== CẤU HÌNH DỰ ÁN (config.yaml) ===
{'seed': 42, 'paths': {'raw': 'data/raw', 'interim': 'data/interim', 'processed': 'data/processed', 'splits': 'data/splits', 'real_world': 'data/real_world', 'reports': 'reports', 'models': 'models', 'db': 'db/tweets.db'}, 'split': {'train': 0.7, 'val': 0.15, 'test': 0.15}, 'tfidf': {'ngram_range': [1, 2], 'min_df': 3, 'max_df': 0.9, 'max_features': 50000, 'sublinear_tf': True}, 'cv': {'folds': 5}, 'primary_metric': 'f1_pos', 'balance': 'class_weight'}

=== KIỂM TRA HÀM CHE TỪ TỤC (mask_profanity) ===
Trước khi che: This racist bigot called her a b***h and used the word f**k.


### Nhận xét (P0)
- Cấu trúc thư mục dự án, tệp cấu hình `config.yaml`, danh sách thư viện `requirements.txt` và mô-đun tiện ích `src/utils.py` đã được khởi tạo đầy đủ.
- Cell `bootstrap` hoạt động ổn định, tự động nhận diện môi trường Google Colab hoặc máy Local mà không cần sửa đường dẫn thủ công.
- Seed ngẫu nhiên (`seed = 42`) và hàm che từ nhạy cảm `mask_profanity()` đã sẵn sàng cho toàn bộ các giai đoạn P1 → P12.